<a href="https://colab.research.google.com/github/sukanya9020/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sukanya9020/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1. My lane as an ML task

**Lane:** Refresh / Content Opportunity Scoring

**ML task type:** Ranking / Scoring

The goal is to assign each content page a priority score and rank pages from higher to lower
priority for human review.

The decision supported by this score is: **Which pages should the content team review first
for possible refresh, improvement, protection, pruning, or monitoring?**

This is a ranking/scoring problem because the team has limited review capacity and needs to
decide which pages should be considered first, rather than simply assigning every page a
yes/no label.

The output will be used as decision support. A high score means a page is higher in the
review queue; it does not mean that a refresh is guaranteed to improve the page.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

### Target / proxy

For the starter dataset, I will use **`is_declining_label`** as a provisional proxy target.

This label indicates whether the observed content trend is classified as declining in the
available data. It can help us explore whether pages with different observable signals can
be separated or prioritized.

However, this is only a proxy for the real business question. It does **not** mean that the
page will decline in the future, and it does not mean that refreshing the page will recover
its performance.

For a stronger future version of the project, I would prefer a future-looking outcome, such
as whether a page actually shows a measurable improvement or deterioration in a later
observation window.

### Why this target is useful for the starter exercise

The proxy is already available in the starter dataset, which allows us to build and evaluate
an initial approach while keeping the limitations explicit.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

The main success metric I will use is **Precision@K**.

Precision@K measures how many of the top K pages selected by the ranking are relevant
according to the target or proxy.

For example, if the team can review the top 50 pages, Precision@50 tells us what fraction
of those 50 pages match the outcome being evaluated.

This metric matches the real business decision because the content team has limited time and
cannot review every page.

I will also consider **Recall@K** or **Average Precision** as supporting metrics when
comparing different approaches.

The metric should be interpreted as evaluation of the ranking against the available observed
target/proxy, not as proof that refreshing a page will cause improvement.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

### Unit of analysis

**One row represents one pseudonymized content item/page.**

The model or scoring system will operate at the content-page level. Each row contains
observable signals about that page, such as impressions, CTR, average position, content age,
engagement, and trend information.

The page-level unit is appropriate because the action we want to support is also page-level:
deciding which individual pages should be reviewed first.

In [ ]:
!git clone -q https://github.com/sukanya9020/flyrank-ml-internship.git /content/flyrank-ml-internship
print("Repository cloned successfully.")

fatal: destination path '/content/flyrank-ml-internship' already exists and is not an empty directory.
Repository cloned successfully.


In [ ]:
import pandas as pd

# Load the starter dataset
data_path = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

# Create the provisional target/proxy from the observed trend direction
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Select useful page-level columns
unit_df = df[
    [
        "content_id",
        "client_id",
        "content_type",
        "impressions_90d",
        "sessions_90d",
        "ctr",
        "avg_position",
        "content_age_days",
        "days_since_last_update",
        "is_declining_label"
    ]
].copy()

print("One row = one pseudonymized content item/page")
print("Rows:", len(unit_df))
print("Columns:", len(unit_df.columns))

display(unit_df.head(10))

One row = one pseudonymized content item/page
Rows: 30000
Columns: 10


,content_id,client_id,content_type,impressions_90d,sessions_90d,ctr,avg_position,content_age_days,days_since_last_update,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,17,0.76,10.6,187,20,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,9,0.05,20.3,445,25,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,141,20,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,78,0.49,6.2,463,22,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,145,0.13,44.0,263,14,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,5,0.03,8.5,147,20,1
6,content_9a34b442b552,client_8722616204,keyword article,20,1,0.00,7.0,90,20,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,28,0.06,21.2,445,22,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,32574,68,0.09,46.0,90,20,1
9,content_c27558df2b0c,client_19581e27de,keyword article,1240,3,0.16,4.9,257,104,1


In [ ]:
print("Target/proxy column:")
display(
    unit_df[["content_id", "is_declining_label"]].head(10)
)

print("\nTarget/proxy distribution:")
print(unit_df["is_declining_label"].value_counts(dropna=False))

Target/proxy column:


,content_id,is_declining_label
0,content_304f48230142,1
1,content_a1fb4e703a9e,1
2,content_9aa793d4d895,1
3,content_331d6c4de07b,0
4,content_d99b7a2d90ca,1
5,content_d4084a4bc775,1
6,content_9a34b442b552,1
7,content_a63219c6e95a,0
8,content_5e6c160719bc,1
9,content_c27558df2b0c,1



Target/proxy distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

A fixed rule could prioritize pages using one simple condition, such as selecting pages with
low CTR or high content age. However, page performance depends on several signals together,
including impressions, sessions, CTR, average position, content age, freshness, and engagement.

A scoring or ML approach can combine multiple signals and learn patterns in the available
data instead of relying on one manually chosen threshold.

ML is useful here because the goal is to rank pages for limited review capacity. Different
signals may interact, and a ranked score can provide a more flexible review queue than a
single fixed rule.

However, ML does not automatically prove that a page needs a refresh or that a refresh will
improve performance. The output remains decision support for human reviewers.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 6. Self-check

- **ML task type:** Ranking / Scoring
- **Lane:** Refresh / Content Opportunity Scoring
- **Target/proxy:** `is_declining_label`, created from the observed `trend_direction == "down"` signal for the starter exercise.
- **Success metric:** Precision@K, with Recall@K and Average Precision as supporting metrics.
- **Unit of analysis:** One row represents one pseudonymized content item/page.
- **Decision supported:** Which pages should a content team review first when review capacity is limited?
- **Why ML:** Multiple page-level signals can be combined instead of relying on one fixed threshold.
- **Limitation:** The proxy represents an observed current trend and does not prove future decline or that a refresh will cause recovery.
- **Data check:** The notebook loads the starter dataset and displays the page-level dataframe and target/proxy distribution.